# Within-Farm Ordinations: 16S ASV Bacteria (Bagged vs Unbagged)

One ordination panel per farm (inside_forest → full_sun gradient).

**Four-row layout:**
- Row 0: global prevalence filter (>=10% of 294 samples), CLR-PCA, confidence ellipses
- Row 1: same global CLR-PCA, confidence ellipses + top N_ARROWS=4 loading arrows
- Row 2: per-farm prevalence filter (>=10% within farm), CLR-PCA, confidence ellipses
- Row 3: per-farm RPCA (matrix completion + rCLR), confidence ellipses + top N_ARROWS=4 loading arrows

CLR-PCA uses TSS + multiplicative zero-replacement + clr; RPCA uses OptSpace matrix
completion + rCLR and handles zeros without pseudocounts.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.transforms as transforms
from matplotlib.patches import Ellipse
import seaborn as sns
import biom
from pathlib import Path
from sklearn.decomposition import PCA
from skbio.stats.composition import multi_replace, clr
# RPCA components — used only via rpca_dense() wrapper defined in the helpers cell.
# deicode 0.2.4 bug: to_dataframe() without dense=True returns NaN in biom 2.1.x,
# causing rclr() to raise ValueError. rpca_dense() calls to_dataframe(dense=True).
from deicode.preprocessing import rclr
from deicode.matrix_completion import MatrixCompletion
import skbio                      # skbio.OrdinationResults, skbio.stats.distance
from scipy.linalg import svd

warnings.filterwarnings('ignore')

In [ ]:
ROOT     = Path('..').resolve()
BIOM_DIR = ROOT / 'results' / 'biom'
DATA_DIR = ROOT / 'data'
FIG_DIR  = ROOT / 'results' / 'figures' / 'dim_reduction'
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
meta = pd.read_csv(
    DATA_DIR / 'sample_metadata_with_microscopy.txt',
    sep='\t', index_col=0, dtype=str,
)
# Remove the QIIME2 types row if present
meta = meta.drop(index='#q2:types', errors='ignore').copy()
print(f'Metadata: {meta.shape[0]} samples, {meta.shape[1]} columns')

# 16S bacteria: biosamples after decontam and off-target filtering in R
tbl_16s    = biom.load_table(
    str(BIOM_DIR / 'cfm_16s_bacteria_biosamples_feature-table.biom')
)
counts_raw = tbl_16s.to_dataframe(dense=True)  # ASVs x samples
print(f'16S raw: {counts_raw.shape[0]} ASVs x {counts_raw.shape[1]} samples')


In [ ]:
# Farm order: inside_forest -> full_sun deforestation gradient
FARM_LEVELS = ['ib', 'vr', 'sa', 'kk', 'mt', 'vi', 'yb']

PALETTES = {
    # Primary colour: bagged vs unbagged treatment
    'sample_type': {
        'bagged_flower':   '#4C72B0',
        'unbagged_flower': '#DD8452',
    },
    # Derived palettes (consistent with 01_dim_reduction.ipynb)
    'farm_id': dict(zip(
        sorted(meta['farm_id'].dropna().unique()),
        sns.color_palette('Dark2', n_colors=meta['farm_id'].nunique()),
    )),
    'management_type': dict(zip(
        sorted(meta['management_type'].dropna().unique()),
        sns.color_palette('Set1', n_colors=meta['management_type'].nunique()),
    )),
}


In [ ]:
def filter_counts(counts_df, min_prevalence=0.0, min_total=0):
    """Keep features meeting minimum prevalence and total-count thresholds."""
    n_samples  = counts_df.shape[1]
    prevalence = (counts_df > 0).sum(axis=1) / n_samples
    total      = counts_df.sum(axis=1)
    keep       = (prevalence >= min_prevalence) & (total >= min_total)
    return counts_df[keep]


def prepare_clr(counts_df: pd.DataFrame) -> pd.DataFrame:
    """CLR-transform a features x samples count matrix.

    multi_replace applies TSS normalisation then fills zeros with
    delta = 1/N^2 (Martin-Fernandez multiplicative replacement).
    Returns a samples x features DataFrame.
    """
    comp = multi_replace(counts_df.T.values)
    return pd.DataFrame(clr(comp), index=counts_df.columns, columns=counts_df.index)


def confidence_ellipse(x, y, ax, n_std=2.0, **kwargs):
    """Draw a covariance-based confidence ellipse (copied from 01_dim_reduction.ipynb)."""
    if len(x) < 3:
        return
    cov     = np.cov(x, y)
    pearson = cov[0, 1] / np.sqrt(cov[0, 0] * cov[1, 1] + 1e-12)
    ellipse = Ellipse(
        (0, 0),
        width=np.sqrt(1 + pearson) * 2,
        height=np.sqrt(1 - pearson) * 2,
        **kwargs,
    )
    t = (
        transforms.Affine2D()
        .rotate_deg(45)
        .scale(np.sqrt(cov[0, 0]) * n_std, np.sqrt(cov[1, 1]) * n_std)
        .translate(np.mean(x), np.mean(y))
    )
    ellipse.set_transform(t + ax.transData)
    ax.add_patch(ellipse)


def rpca_dense(table, n_components=2, min_sample_count=1,
               min_feature_count=1, max_iterations=5):
    """RPCA wrapper that fixes the deicode 0.2.4 NaN bug.

    deicode.rpca.rpca calls table.to_dataframe() without dense=True.
    biom 2.1.x to_dataframe(dense=False) returns a sparse pandas DataFrame
    where absent entries are NaN, not 0. rclr() detects these NaN values
    and raises ValueError before any transformation. This wrapper is
    identical to deicode.rpca.rpca except it calls to_dataframe(dense=True).
    """
    n_features, n_samples = table.shape

    def obs_filter(val, id_, md):
        return sum(val) > min_feature_count

    def samp_filter(val, id_, md):
        return sum(val) > min_sample_count

    def freq_filter(val, id_, md):
        return (np.sum(val > 0) / n_samples) > 0.0

    table = table.filter(obs_filter,  axis='observation', inplace=False)
    table = table.filter(freq_filter, axis='observation', inplace=False)
    table = table.filter(samp_filter, axis='sample',      inplace=False)

    # KEY FIX: dense=True → 0 for absent entries instead of NaN
    dense = table.to_dataframe(dense=True).T  # shape: samples x features

    opt = MatrixCompletion(n_components=n_components,
                           max_iterations=max_iterations).fit(rclr(dense))
    n_components = opt.s.shape[0]
    rename_cols  = ['PC' + str(i + 1) for i in range(n_components)]

    X = opt.sample_weights @ opt.s @ opt.feature_weights.T
    X = X - X.mean(axis=0)
    X = X - X.mean(axis=1).reshape(-1, 1)
    u, s, v = svd(X)
    u = u[:, :n_components]
    v = v.T[:, :n_components]
    p = s**2 / np.sum(s**2)
    p, s = p[:n_components], s[:n_components]

    feature_loading      = pd.DataFrame(v, index=dense.columns, columns=rename_cols)
    sample_loading       = pd.DataFrame(u, index=dense.index,   columns=rename_cols)
    proportion_explained = pd.Series(p, index=rename_cols)
    eigvals              = pd.Series(s, index=rename_cols)

    # deicode adds a zero PC3 for Emperor compatibility when n_components=2
    if n_components == 2:
        feature_loading['PC3']          = 0
        sample_loading['PC3']           = 0
        eigvals.loc['PC3']              = 0
        proportion_explained.loc['PC3'] = 0

    ord_res = skbio.OrdinationResults(
        'rpca_biplot', '(Robust Aitchison) RPCA Biplot',
        eigvals.copy(),
        samples=sample_loading.copy(),
        features=feature_loading.copy(),
        proportion_explained=proportion_explained.copy())
    dist_res = skbio.stats.distance.DistanceMatrix(
        opt.distance, ids=sample_loading.index)

    return ord_res, dist_res

In [ ]:
# Prevalence threshold applied globally across all 294 samples
# — ensures every farm panel uses the same feature set
PREV_THRESHOLD_16S = 0.10

counts_f = filter_counts(counts_raw, PREV_THRESHOLD_16S)
print(f'16S ASV bacteria: {counts_raw.shape[0]} -> {counts_f.shape[0]} features after prevalence >= {PREV_THRESHOLD_16S:.0%}')
print(f'Samples in table: {counts_f.shape[1]}')


In [ ]:
def parse_taxon(taxon_str: str, n_levels: int = 2) -> str:
    """Return the deepest n non-empty taxonomy levels as a readable label."""
    parts = [p.strip() for p in taxon_str.split(';')]
    labeled = []
    for p in parts:
        if '__' in p:
            prefix, val = p.split('__', 1)
            val = val.strip()
            if prefix.lower() == 'sh':  # skip UNITE SH codes
                continue
            if val and val not in ('NA', 'unidentified', 'uncultured', ''):
                labeled.append(val)
    if not labeled:
        return taxon_str[:30]
    return ' '.join(labeled[-n_levels:]).replace('_', ' ')

tax = (
    pd.read_csv(BIOM_DIR / 'cfm_16s_bacteria_biosamples_taxonomy.tsv',
                sep='\t', index_col=0)
    .assign(label=lambda df: df['Taxon'].apply(lambda x: parse_taxon(str(x))))
    ['label']
)
print(f'Taxonomy: {len(tax)} entries | example: {tax.iloc[0]}')


In [ ]:
farm_rpca_results = {}

for farm in FARM_LEVELS:
    farm_ids = [
        s for s in meta.index[
            (meta['farm_id'] == farm) &
            (meta['sample_type'].isin(PALETTES['sample_type'].keys()))
        ]
        if s in tbl_16s.ids('sample')
    ]
    tbl_farm = tbl_16s.filter(farm_ids, axis='sample', inplace=False)
    ordination, _ = rpca_dense(tbl_farm)
    farm_rpca_results[farm] = {'ordination': ordination, 'meta': meta.loc[farm_ids]}
    ev = ordination.proportion_explained.values
    print(f'{farm}: n={len(farm_ids)}  RPCA1={ev[0]:.1%}  RPCA2={ev[1]:.1%}')

In [ ]:
farm_pca_results = {}  # populated in this loop; used by the loadings cell

farm_mgmt = meta.groupby('farm_id')['management_type'].first()

# N_ARROWS   : number of feature arrows in the biplot rows (rows 1 and 3).
#              Increase (e.g. 4 → 12) to show more features; labels may overlap.
# ARROW_SCALE: percentile of sample score cloud used to set arrow length.
#              Increase (85 → 95) for longer arrows.
N_ARROWS    = 4
ARROW_SCALE = 85

# 4 rows x 7 columns
# Row 0: global prevalence filter + CLR-PCA, no arrows
# Row 1: global prevalence filter + CLR-PCA + feature loading arrows
# Row 2: per-farm prevalence filter + CLR-PCA, no arrows
# Row 3: per-farm RPCA (matrix completion + rCLR) + feature loading arrows
fig, axes = plt.subplots(4, 7, figsize=(28, 20))
fig.suptitle(
    'Within-farm ordinations: bagged vs unbagged flowers (16S ASV bacteria)\n'
    'Row 0: CLR-PCA  |  Row 1: CLR-PCA + arrows  |  Row 2: per-farm CLR-PCA  |  Row 3: RPCA (OptSpace + rCLR) + arrows',
    fontsize=12, fontweight='bold',
)

for ax_idx, farm in enumerate(FARM_LEVELS):

    farm_ids = meta.index[
        (meta['farm_id'] == farm) &
        (meta['sample_type'].isin(PALETTES['sample_type'].keys()))
    ]
    farm_ids_global = farm_ids[farm_ids.isin(counts_f.columns)]
    n_samples = len(farm_ids_global)
    mgmt = farm_mgmt.get(farm, '')

    # ---- Row 0: global filter, no arrows ----
    ax0 = axes[0, ax_idx]
    if ax_idx == 0:
        ax0.set_ylabel('Global filter\n(>=10% of 294)', fontsize=8, labelpad=8)

    if n_samples < 3:
        ax0.set_visible(False)
    else:
        counts_farm_global = counts_f[farm_ids_global]
        clr_farm_global    = prepare_clr(counts_farm_global)
        pca0 = PCA(n_components=2)
        pcs0 = pca0.fit_transform(clr_farm_global)
        ev0  = pca0.explained_variance_ratio_ * 100

        farm_pca_results[farm] = {
            'pca':         pca0,
            'feature_ids': clr_farm_global.columns.tolist(),
            'ev':          ev0,
        }

        n_feats_global = counts_farm_global.shape[0]
        sample_types0  = meta.loc[farm_ids_global, 'sample_type']

        for val, color in PALETTES['sample_type'].items():
            mask = (sample_types0 == val).values
            if mask.sum() < 1:
                continue
            lbl = val.replace('_', ' ') if ax_idx == 0 else '_nolegend_'
            ax0.scatter(pcs0[mask, 0], pcs0[mask, 1],
                        c=color, s=30, alpha=0.8,
                        edgecolors='white', linewidths=0.3, label=lbl)
            if mask.sum() >= 3:
                confidence_ellipse(pcs0[mask, 0], pcs0[mask, 1], ax0,
                                   n_std=2.0, facecolor='none',
                                   edgecolor=color, linewidth=1.5, alpha=0.8)
                ax0.plot(pcs0[mask, 0].mean(), pcs0[mask, 1].mean(),
                         marker='x', color=color, markersize=8, markeredgewidth=2)

        ax0.set_title(f'{farm} | {mgmt}\nn={n_samples} | {n_feats_global} features (global)', fontsize=8)
        ax0.set_xlabel(f'PC1 ({ev0[0]:.1f}%)', fontsize=7)
        ax0.set_ylabel(f'PC2 ({ev0[1]:.1f}%)', fontsize=7)
        ax0.tick_params(labelsize=6)
        if ax_idx == 0:
            ax0.legend(fontsize=7, title='sample type', title_fontsize=7, framealpha=0.8)

    # ---- Row 1: global filter + feature loading arrows ----------------------
    ax1 = axes[1, ax_idx]
    if ax_idx == 0:
        ax1.set_ylabel('Global filter\n+ loading arrows', fontsize=8, labelpad=8)

    if n_samples < 3:
        ax1.set_visible(False)
    else:
        loadings    = pca0.components_[:2].T  # shape (n_features, 2); no re-fit needed
        feature_ids = farm_pca_results[farm]['feature_ids']

        for val, color in PALETTES['sample_type'].items():
            mask = (sample_types0 == val).values
            if mask.sum() < 1:
                continue
            ax1.scatter(pcs0[mask, 0], pcs0[mask, 1],
                        c=color, s=30, alpha=0.8,
                        edgecolors='white', linewidths=0.3, label='_nolegend_')
            if mask.sum() >= 3:
                confidence_ellipse(pcs0[mask, 0], pcs0[mask, 1], ax1,
                                   n_std=2.0, facecolor='none',
                                   edgecolor=color, linewidth=1.5, alpha=0.8)
                ax1.plot(pcs0[mask, 0].mean(), pcs0[mask, 1].mean(),
                         marker='x', color=color, markersize=8, markeredgewidth=2)

        # N_ARROWS   : arrows per panel ranked by L2 norm across PC1+PC2
        # ARROW_SCALE: percentile of sample score cloud for arrow length calibration
        l2_norms = np.linalg.norm(loadings, axis=1)
        top_idx  = np.argsort(l2_norms)[-N_ARROWS:]
        scale    = np.percentile(np.abs(pcs0[:, :2]), ARROW_SCALE)
        feat_max = np.abs(loadings[top_idx]).max()

        for i in top_idx:
            lx = loadings[i, 0] / feat_max * scale
            ly = loadings[i, 1] / feat_max * scale
            ax1.annotate('', xy=(lx, ly), xytext=(0, 0),
                         arrowprops=dict(arrowstyle='->', color='#555555', lw=0.8, alpha=0.8))
            label = tax.get(feature_ids[i], feature_ids[i][:10])
            ax1.text(lx * 1.08, ly * 1.08, label,
                     fontsize=4.5, ha='center', va='center', color='#333333')

        ax1.axhline(0, color='grey', lw=0.4, ls='--', alpha=0.4)
        ax1.axvline(0, color='grey', lw=0.4, ls='--', alpha=0.4)
        ax1.set_title(f'{farm} | {mgmt}\nn={n_samples} | {n_feats_global} features (global, arrows)', fontsize=8)
        ax1.set_xlabel(f'PC1 ({ev0[0]:.1f}%)', fontsize=7)
        ax1.set_ylabel(f'PC2 ({ev0[1]:.1f}%)', fontsize=7)
        ax1.tick_params(labelsize=6)

    # ---- Row 2: per-farm filter ----
    ax2 = axes[2, ax_idx]
    if ax_idx == 0:
        ax2.set_ylabel('Per-farm filter\n(>=10% within farm)', fontsize=8, labelpad=8)

    farm_ids_raw = farm_ids[farm_ids.isin(counts_raw.columns)]
    n_raw        = len(farm_ids_raw)

    if n_raw < 3:
        ax2.set_visible(False)
    else:
        counts_farm_local = filter_counts(counts_raw[farm_ids_raw], PREV_THRESHOLD_16S)
        clr_farm_local    = prepare_clr(counts_farm_local)
        pca1 = PCA(n_components=2)
        pcs1 = pca1.fit_transform(clr_farm_local)
        ev1  = pca1.explained_variance_ratio_ * 100

        n_feats_local = counts_farm_local.shape[0]
        sample_types1 = meta.loc[farm_ids_raw, 'sample_type']

        for val, color in PALETTES['sample_type'].items():
            mask = (sample_types1 == val).values
            if mask.sum() < 1:
                continue
            ax2.scatter(pcs1[mask, 0], pcs1[mask, 1],
                        c=color, s=30, alpha=0.8,
                        edgecolors='white', linewidths=0.3, label='_nolegend_')
            if mask.sum() >= 3:
                confidence_ellipse(pcs1[mask, 0], pcs1[mask, 1], ax2,
                                   n_std=2.0, facecolor='none',
                                   edgecolor=color, linewidth=1.5, alpha=0.8)
                ax2.plot(pcs1[mask, 0].mean(), pcs1[mask, 1].mean(),
                         marker='x', color=color, markersize=8, markeredgewidth=2)

        ax2.set_title(f'{farm}\nn={n_raw} | {n_feats_local} features (per-farm)', fontsize=8)
        ax2.set_xlabel(f'PC1 ({ev1[0]:.1f}%)', fontsize=7)
        ax2.set_ylabel(f'PC2 ({ev1[1]:.1f}%)', fontsize=7)
        ax2.tick_params(labelsize=6)

    # ---- Row 3: RPCA (matrix completion + rCLR) + ellipses + arrows ---------
    ax3 = axes[3, ax_idx]
    if ax_idx == 0:
        ax3.set_ylabel('RPCA (OptSpace + rCLR) + arrows', fontsize=8, labelpad=8)

    res3 = farm_rpca_results.get(farm)
    if res3 is None:
        ax3.set_visible(False)
    else:
        ord3    = res3['ordination']
        meta3   = res3['meta']
        ev3     = ord3.proportion_explained.values
        scores3 = ord3.samples          # DataFrame indexed by sample_id; cols PC1, PC2, PC3
        feat3   = ord3.features         # DataFrame indexed by feature_id

        for val, color in PALETTES['sample_type'].items():
            mask    = (meta3['sample_type'] == val).values
            ids_sel = meta3.index[mask]
            pts     = scores3.loc[ids_sel, ['PC1', 'PC2']].values
            if len(pts) < 1:
                continue
            ax3.scatter(pts[:, 0], pts[:, 1],
                        c=color, s=30, alpha=0.8,
                        edgecolors='white', linewidths=0.3, label='_nolegend_')
            if len(pts) >= 3:
                confidence_ellipse(pts[:, 0], pts[:, 1], ax3,
                                   n_std=2.0, facecolor='none',
                                   edgecolor=color, linewidth=1.5, alpha=0.8)
                ax3.plot(pts[:, 0].mean(), pts[:, 1].mean(),
                         marker='x', color=color, markersize=8, markeredgewidth=2)

        # Loading arrows — same N_ARROWS and ARROW_SCALE constants as Row 1
        feat_arr = feat3[['PC1', 'PC2']].values
        l2_norms = np.linalg.norm(feat_arr, axis=1)
        top_idx  = np.argsort(l2_norms)[-N_ARROWS:]
        scale    = np.percentile(np.abs(scores3[['PC1', 'PC2']].values), ARROW_SCALE)
        feat_max = np.abs(feat_arr[top_idx]).max()
        feat_ids = feat3.index.tolist()
        for i in top_idx:
            lx = feat_arr[i, 0] / feat_max * scale
            ly = feat_arr[i, 1] / feat_max * scale
            ax3.annotate('', xy=(lx, ly), xytext=(0, 0),
                         arrowprops=dict(arrowstyle='->', color='#555555', lw=0.8, alpha=0.8))
            label = tax.get(feat_ids[i], feat_ids[i][:10])
            ax3.text(lx * 1.08, ly * 1.08, label,
                     fontsize=4.5, ha='center', va='center', color='#333333')

        ax3.axhline(0, color='grey', lw=0.4, ls='--', alpha=0.4)
        ax3.axvline(0, color='grey', lw=0.4, ls='--', alpha=0.4)
        ax3.set_title(f'{farm} | {mgmt}\nn={len(meta3)} | {feat3.shape[0]} features (RPCA)', fontsize=8)
        # RPCA proportion_explained is in [0,1] — use :.1% not :.1f%
        ax3.set_xlabel(f'RPCA1 ({ev3[0]:.1%})', fontsize=7)
        ax3.set_ylabel(f'RPCA2 ({ev3[1]:.1%})', fontsize=7)
        ax3.tick_params(labelsize=6)

plt.tight_layout(rect=[0, 0, 1, 0.95])  # top 5% reserved for 2-line suptitle
fig.savefig(FIG_DIR / '16s_pca_by_farm.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
n_top = 10  # top features per PC per farm; adjust if labels are too crowded

# bar height per feature + padding; scales total figure height with n_top
fig, axes = plt.subplots(7, 2, figsize=(14, n_top * 7 * 0.35 + 7))
fig.suptitle('PCA Loadings per farm — 16S ASV bacteria',
             fontsize=13, fontweight='bold')

POS_COLOR = 'steelblue'  # positive loading bar colour
NEG_COLOR = 'tomato'  # negative loading bar colour

for row_idx, farm in enumerate(FARM_LEVELS):
    result      = farm_pca_results[farm]
    pca_obj     = result['pca']
    feature_ids = result['feature_ids']
    ev          = result['ev']

    for col_idx, pc_idx in enumerate([0, 1]):
        ax       = axes[row_idx, col_idx]
        loadings = pca_obj.components_[pc_idx]
        top_idx  = np.argsort(np.abs(loadings))[-n_top:]
        vals     = loadings[top_idx]
        names    = [
            tax.get(feature_ids[i], feature_ids[i][:12])
            + f' [{feature_ids[i][:6]}]'
            for i in top_idx
        ]
        colors   = [POS_COLOR if v >= 0 else NEG_COLOR for v in vals]
        ax.barh(names, vals, color=colors)
        ax.axvline(0, color='black', lw=0.8)
        ax.set_xlabel('Loading value', fontsize=7)
        ax.set_title(
            f'{farm} | PC{pc_idx + 1} ({ev[pc_idx]:.1f}%)',
            fontsize=8, fontweight='bold'
        )
        ax.tick_params(axis='y', labelsize=6)  # taxonomy label size
        ax.tick_params(axis='x', labelsize=7)

# rect top reserves space for suptitle and prevents overlap with top row
plt.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(FIG_DIR / '16s_pca_loadings_by_farm.pdf', dpi=300, bbox_inches='tight')
plt.show()
